# Exercise 2: PyTorch core

In this exercise you’ll build core PyTorch “muscle memory” that you’ll reuse in basically every model you write:

- **Autograd**: how gradients are created, how they accumulate, and how to compute gradients for one or multiple inputs.
- **Dataloading**: writing small `Dataset`s, using `DataLoader`, and custom `collate_fn`.
- **Optimizers**: implementing **AdamW** updates from scratch (state, bias correction, weight decay).
- **Training basics**: a clean single training step.
- **Initialization**: fan-in/out and common initializers (Xavier / Kaiming), plus a helper to init `nn.Linear`.

As before: fill in all `TODO`s without changing function names or signatures.
When debugging, print shapes/dtypes/devices, and write tiny sanity checks (e.g. compare to PyTorch’s built-ins).


In [2]:
from __future__ import annotations
from dataclasses import dataclass
import torch
from torch import nn

## Autograd fundamentals

PyTorch builds a computation graph when you apply operations to tensors with `requires_grad=True`.
Calling `backward()` (or `torch.autograd.grad`) computes gradients by traversing that graph.

### Key concepts
- **Leaf tensor**: a tensor created by you (not the result of an operation) with `requires_grad=True`. Leaf tensors can store gradients in `.grad`.
- **Gradient accumulation**: calling `backward()` adds into `.grad` (it does not overwrite). You must reset gradients between steps/calls.
- **`torch.autograd.grad` vs `.backward()`**
  - `torch.autograd.grad(f, x)` returns `df/dx` directly and does not write into `x.grad` unless you explicitly do so.
  - `f.backward()` writes gradients into `.grad` of leaf tensors.

In the next functions you’ll compute gradients for a simple scalar function such as `f(x) = sum(x^2)` using both APIs.

### `torch.no_grad()`
Wrap inference-only code to avoid tracking gradients and building graphs:
- saves memory
- speeds up evaluation

### `detach()`
`y = x.detach()` returns a tensor that shares data with `x` but is **not connected** to the autograd graph.
This is useful when you want to treat something as a constant target.

### `model.train()` vs `model.eval()`
- `train()` enables training behavior (e.g. dropout active, batchnorm updates running stats).
- `eval()` enables inference behavior (e.g. dropout off, batchnorm uses running stats).

In [3]:
def grad_with_autograd_grad(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using torch.autograd.grad

    Requirements:
    - Do not call .backward().
    - x should require grad inside the function (don't assume it does).
    - Must return df/dx
    """
    func = torch.sum(x**2)
    return torch.autograd.grad(func, x)
    

x = torch.tensor([1, 2, 3], dtype=torch.float32, requires_grad=True)
print(grad_with_autograd_grad(x))
# f(x) = x_1^2 + x_2^2 + x_3^2
# df(x)/x = df(x)/x_1, df(x)/x_2, df(x)/x_3
# df(x)/x_1 = 2x_1
# df(x)/x_2 = 2x_2
# df(x)/x_3 = 2x_3
# df(x)/x = 2, 4, 6

(tensor([2., 4., 6.]),)


In [4]:
def grad_with_backward(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using .backward().

    Requirements:
    - Must return df/dx
    - Must not leak gradients across calls (watch x.grad accumulation)
    """
    if x.grad is not None:
        x.grad.zero_()
    
    x_leaf = x.detach().requires_grad_(True)
    
    y = (x_leaf ** 2).sum()
    
    y.backward()
    
    return x_leaf.grad.detach()

x = torch.tensor([1, 2, 3], dtype=torch.float32, requires_grad=True)
print(grad_with_autograd_grad(x))

(tensor([2., 4., 6.]),)


In [5]:
def grad_wrt_multiple_inputs(
    a: torch.Tensor, b: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute gradients w.r.t. multiple inputs. The function is f(a, b) = sum(a^2 + ab).

    Return:
        (df/da, df/db)

    Requirements:
    - Use torch.autograd.grad
    - Ensure both a and b require grad in this function.
    """
    if a.grad is not None:
        a.grad.zero_()

    if b.grad is not None:
        b.grad.zero_()

    a_leaf = a.detach().requires_grad_(True)
    b_leaf = b.detach().requires_grad_(True)

    func = (a_leaf**2 + a_leaf*b_leaf).sum()

    grad_a, grad_b = torch.autograd.grad(func, (a_leaf, b_leaf))

    return (grad_a.detach(), grad_b.detach())

a = torch.tensor([1, 2, 3], dtype=torch.float32, requires_grad=True)
b = torch.tensor([5, 6, 7], dtype=torch.float32, requires_grad=True)
grad_wrt_multiple_inputs(a, b)
# f(a, b) = sum(a^2 + ab)
# f(a, b) = (a_1^2 + a_1*b_1) + (a_2^2 + a_2*b_2) + (a_3^2 + a_3*b_3)
# df(a, b)/a_1 = 2a_1 + b_1
# df(a, b)/a_2 = 2a_2 + b_2
# df(a, b)/a_3 = 2a_3 + b_3
# df(a, b)/a =  7, 10, 13
# df(a, b)/b_1 = a_1
# df(a, b)/b_2 = a_2
# df(a, b)/b_3 = a_3
# df(a, b)/b = 1, 2, 3

(tensor([ 7., 10., 13.]), tensor([1., 2., 3.]))

## Dataloading

In PyTorch, a `Dataset` defines how to fetch a *single* training example, and a `DataLoader` handles:
- batching
- shuffling
- parallel workers
- optional custom batching logic via `collate_fn`

### `Dataset` in one sentence
A `Dataset` only needs:
- `__len__`: number of items
- `__getitem__`: return one item (e.g. `(x, y)`)

### Why `collate_fn` matters
The default DataLoader collation stacks items along a new batch dimension.
That works for fixed-size tensors, but it breaks for **variable-length sequences**.

So we’ll implement padding ourselves:
- Convert a list of 1D token sequences into a padded tensor `(B, T_max)`
- Track `lengths` and a `padding_mask`

### Mask convention for padding
For padding masks in this exercise:
- `padding_mask[b, t] == True` means **this is padding / invalid**
- `padding_mask[b, t] == False` means **this is a real token**

In [6]:
from torch.utils.data import DataLoader, Dataset

In [7]:
class TensorPairDataset(Dataset):
    """
    Minimal dataset wrapping (x, y).

    x: (N, ...)
    y: (N, ...)

    N is the number of samples. The dataset should return tuples of (x[i], y[i]).
    """

    def __init__(self, x: torch.Tensor, y: torch.Tensor):
        self.x = x
        self.y = y

    def __len__(self) -> int:
        return self.x.shape[0]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        assert 0 <= idx < self.x.shape[0]

        return (self.x[idx], self.y[idx])

x = torch.tensor([1, 2, 3, 4, 5, 6, 7], dtype=torch.int32)
y = torch.tensor([1, 1, 0, 0, 0, 1, 1], dtype=torch.int32)

myDataset = TensorPairDataset(x, y)
print(len(myDataset))
print(myDataset[0])
print(myDataset[3])

7
(tensor(1, dtype=torch.int32), tensor(1, dtype=torch.int32))
(tensor(4, dtype=torch.int32), tensor(0, dtype=torch.int32))


In [8]:
class NextTokenDataset(Dataset):
    """
    Next-token prediction dataset.

    Given tokens of shape (N, T), produce:
      input_ids  = tokens[:, :-1]
      target_ids = tokens[:, 1:]

    Return per item:
      (input_ids, target_ids)

    Notes:
    - Returned tensors should be 1D of length (T-1).
    - dtype should remain integer.
    """

    def __init__(self, tokens: torch.Tensor):
        self.tokens = tokens
        self.input_ids = self.tokens[:, :-1]
        self.target_ids = self.tokens[:, 1:]

    def __len__(self) -> int:
        return self.input_ids.shape[0]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        assert 0 <= idx < self.__len__()

        return (self.input_ids[idx], self.target_ids[idx])

tokens = torch.tensor([[1, 2, 3, 4, 5, 6, 7],
                       [3, 3, 4, 5, 6, 1, 8],
                       [2, 3, 1, 2, 8, 3, 2]], dtype=torch.int32)
nextTokenDataset = NextTokenDataset(tokens)

print(len(nextTokenDataset))
print(nextTokenDataset[2])

3
(tensor([2, 3, 1, 2, 8, 3], dtype=torch.int32), tensor([3, 1, 2, 8, 3, 2], dtype=torch.int32))


In [9]:

class RandomCropSequenceDataset(Dataset):
    """
    Sequence dataset that returns random crops of fixed length.

    tokens: (N, T_total)
    crop_len: L

    For each __getitem__:
      - sample a start index s so that s+L <= T_total
      - return tokens[idx, s:s+L]

    Requirements:
    - Use a torch.Generator for deterministic behavior if seed is provided.
    - Do NOT use Python's random module.
    """

    def __init__(self, tokens: torch.Tensor, crop_len: int, seed: int | None = None):
        self.tokens = tokens
        self.crop_len = crop_len
        self.seed = seed
        self.generator = torch.Generator()
        self.generator.manual_seed(seed)

    def __len__(self) -> int:
        return self.tokens.shape[0]

    def __getitem__(self, idx: int) -> torch.Tensor:
        s = torch.randint(low=0, high=self.tokens.shape[-1] - self.crop_len, size=(1,))
        assert 0 <= idx < self.__len__()
        print(f's: {s}')
        return self.tokens[idx, s:s+self.crop_len]

tokens = torch.tensor([[1, 2, 3, 4, 5, 6, 7],
                       [3, 3, 4, 5, 6, 1, 8],
                       [2, 3, 1, 2, 8, 3, 2]], dtype=torch.int32)
randomDataset = RandomCropSequenceDataset(tokens, 3, 42)

print(len(randomDataset))
print(randomDataset[2])

3
s: tensor([3])
tensor([2, 8, 3], dtype=torch.int32)


In [10]:
@dataclass(frozen=True)
class PaddedBatch:
    """
    A padded batch for variable-length sequences.

    tokens: LongTensor (B, T_max)
    lengths: LongTensor (B,)
    padding_mask: BoolTensor (B, T_max) where True means "this is padding"
    """

    tokens: torch.Tensor
    lengths: torch.Tensor
    padding_mask: torch.Tensor


def pad_1d_sequences(seqs: list[torch.Tensor], pad_value: int = 0) -> PaddedBatch:
    """
    Pad a list of 1D integer tensors to the same length.

    Requirements:
    - Return PaddedBatch(tokens, lengths, padding_mask)
    - padding_mask[b, t] == True iff t >= lengths[b]
    - tokens should be dtype long, if not cast them
    """
    # Way 1 of doing it
    T_max = max([seq.shape[0] for seq in seqs])
    lengths = [seq.shape[0] for seq in seqs]
    output = [torch.cat([seq, torch.ones(T_max - length)*pad_value]) for seq, length in zip(seqs, lengths)]
    output = torch.stack(output, dim=0)
    output = output.to(torch.float64)

    padding_mask = (output == pad_value)

    # Way 2 of doing it
    device = seqs[0].device
    
    lengths = torch.tensor([seq.size(0) for seq in seqs], dtype=torch.long, device=device)
    max_len = lengths.max().item()

    seqs_long = [seq.to(torch.long) if seq.dtype != torch.long else seq for seq in seqs]
    tokens = torch.nn.utils.rnn.pad_sequence(
        seqs_long, batch_first=True, padding_value=pad_value
    )

    arange_t = torch.arange(max_len, device=device).unsqueeze(0)
    padding_mask = arange_t >= lengths.unsqueeze(1)

    return PaddedBatch(output, lengths, padding_mask)


tokens = [
    torch.tensor([1, 2, 3, 4]),
    torch.tensor([1, 2]),
    torch.tensor([3, 4, 5, 5, 6, 7, 8])
]
paddedBatch = pad_1d_sequences(tokens)
print('Tokens')
print(paddedBatch.tokens)
print('Lengths')
print(paddedBatch.lengths)
print('Padding Mask')
print(paddedBatch.padding_mask)

Tokens
tensor([[1., 2., 3., 4., 0., 0., 0.],
        [1., 2., 0., 0., 0., 0., 0.],
        [3., 4., 5., 5., 6., 7., 8.]], dtype=torch.float64)
Lengths
tensor([4, 2, 7])
Padding Mask
tensor([[False, False, False, False,  True,  True,  True],
        [False, False,  True,  True,  True,  True,  True],
        [False, False, False, False, False, False, False]])


In [11]:
def collate_next_token_batch(
    batch: list[tuple[torch.Tensor, torch.Tensor]], pad_value: int = 0
) -> dict[str, torch.Tensor]:
    """
    Collate for NextTokenDataset samples that may have variable lengths.

    batch: list of (input_ids, target_ids), each 1D

    Return dict with:
      - input_ids: (B, T_max)
      - target_ids: (B, T_max)
      - attention_mask: (B, T_max) where True means "keep" (NOT padding)
      - padding_mask: (B, T_max) where True means "padding"

    Requirements:
    - pad input_ids and target_ids consistently
    - attention_mask is the logical NOT of padding_mask
    """
    # Implementation 1
    lengths = torch.tensor([max(input.size(0), target.size(0)) for input, target in batch])
    T_max = torch.max(lengths).item()

    inputs = [input for input, _ in batch]
    targets = [target for _, target in batch]

    input_ids = [torch.cat([input_id, torch.ones(T_max - input_id.size(0))*pad_value]) for input_id in inputs]
    input_ids = torch.stack(input_ids, dim=0)
    input_ids = input_ids.to(torch.int32)
    target_ids = [torch.cat([target_id, torch.ones(T_max - target_id.size(0))*pad_value]) for target_id in targets]
    target_ids = torch.stack(target_ids, dim=0)
    target_ids = target_ids.to(torch.int32)

    attention_mask = (input_ids != pad_value)
    padding_mask = (input_ids == pad_value)

    output = {
        'input_ids': input_ids,
        'target_ids': target_ids,
        'attention_mask': attention_mask,
        'padding_mask': padding_mask
    }

    # return output

    # Implementation 2
    nputs, targets = zip(*batch)
    
    device = inputs[0].device
    
    lengths = torch.tensor([inp.size(0) for inp in inputs], dtype=torch.long, device=device)
    max_len = lengths.max().item()

    input_ids = torch.nn.utils.rnn.pad_sequence(
        inputs, batch_first=True, padding_value=pad_value
    ).to(torch.long)
    
    target_ids = torch.nn.utils.rnn.pad_sequence(
        targets, batch_first=True, padding_value=pad_value
    ).to(torch.long)

    arange_t = torch.arange(max_len, device=device).unsqueeze(0)
    
    attention_mask = arange_t < lengths.unsqueeze(1)
    padding_mask = ~attention_mask

    return {
        "input_ids": input_ids,
        "target_ids": target_ids,
        "attention_mask": attention_mask,
        "padding_mask": padding_mask,
    }

batch = [
   (torch.tensor([1, 2, 3, 4], dtype=torch.float32), torch.tensor([2, 3, 4, 5], dtype=torch.float32)),
   (torch.tensor([1], dtype=torch.float32), torch.tensor([1], dtype=torch.float32)),
   (torch.tensor([1, 2, 3], dtype=torch.float32), torch.tensor([2, 3, 4], dtype=torch.float32)),
   (torch.tensor([1, 2, 3, 4, 4, 5], dtype=torch.float32), torch.tensor([1, 2, 3, 4, 4, 5], dtype=torch.float32))
]

collate_next_token_batch(batch)

{'input_ids': tensor([[1, 2, 3, 4, 0, 0],
         [1, 0, 0, 0, 0, 0],
         [1, 2, 3, 0, 0, 0],
         [1, 2, 3, 4, 4, 5]]),
 'target_ids': tensor([[2, 3, 4, 5, 0, 0],
         [1, 0, 0, 0, 0, 0],
         [2, 3, 4, 0, 0, 0],
         [1, 2, 3, 4, 4, 5]]),
 'attention_mask': tensor([[ True,  True,  True,  True, False, False],
         [ True, False, False, False, False, False],
         [ True,  True,  True, False, False, False],
         [ True,  True,  True,  True,  True,  True]]),
 'padding_mask': tensor([[False, False, False, False,  True,  True],
         [False,  True,  True,  True,  True,  True],
         [False, False, False,  True,  True,  True],
         [False, False, False, False, False, False]])}

In [12]:
def make_dataloader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool = True,
    drop_last: bool = False,
    collate_fn=None,
    num_workers: int = 0,
) -> DataLoader:
    """
    Create a DataLoader with optional collate_fn.
    """
    return DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        collate_fn=collate_fn,
        num_workers=num_workers
    )

## Optimizers (AdamW from scratch)

PyTorch optimizers keep **state** for each parameter (e.g. moment estimates in Adam).
In this section you’ll implement **AdamW**, which is Adam + *decoupled* weight decay.

### AdamW state
For each parameter tensor `p` we store:
- `m`: first moment (EMA of gradients)
- `v`: second moment (EMA of squared gradients)
- `t`: step counter

### Update overview (high level)
1) Update moments `m, v`
2) Bias-correct them (`m_hat, v_hat`)
3) Apply parameter update:
   `p -= lr * ( m_hat / (sqrt(v_hat) + eps) + weight_decay * p )`

Notes:
- This update is **in-place** (mutates `p`).
- Gradients should not be modified.
- State tensors must match parameter shape/device/dtype.

In [13]:
@dataclass
class AdamWState:
    """
    Per-parameter AdamW state.

    m: first moment
    v: second moment
    t: step count
    """

    m: torch.Tensor
    v: torch.Tensor
    t: int


def init_adamw_state(p: torch.Tensor) -> AdamWState:
    """
    Initialize AdamW state tensors for a parameter tensor p.

    What to create:
    - m: zeros like p, same shape/device/dtype
    - v: zeros like p, same shape/device/dtype
    - t: step counter starting at 0

    Notes / requirements:
    - Use torch.zeros_like(p) for m and v.
    - Do NOT attach gradients to the state (initialize under torch.no_grad()).
    - t starts at 0. In adamw_step_, increment t to 1 on the first update *before*
      computing bias correction terms (1 - beta1^t) and (1 - beta2^t).
    - State tensors must live on the same device as p (CPU vs GPU) and have the
      same dtype as p.
    """
    with torch.no_grad():
      m = torch.zeros_like(p)
      v = torch.zeros_like(p)
      t = torch.tensor(0, dtype=torch.long, device=p.device)

    return AdamWState(m, v, t)


In [21]:
def adamw_step_(
    p: torch.Tensor,
    grad: torch.Tensor,
    state: AdamWState,
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> AdamWState:
    """
    In-place AdamW parameter update (updates p).

    Algorithm (AdamW):
      m = beta1*m + (1-beta1)*grad
      v = beta2*v + (1-beta2)*grad^2
      m_hat = m / (1 - beta1^t)
      v_hat = v / (1 - beta2^t)
      p = p - lr * (m_hat / (sqrt(v_hat) + eps) + weight_decay * p)

    Requirements:
    - Update p in-place.
    - Return updated state (with incremented t).
    - Do not modify grad.
    - Should work for any tensor shape.
    """
    beta1, beta2 = betas

    with torch.no_grad():
        t = state.t + 1

        state.m.mul_(beta1).add_(grad, alpha=1.0-beta1)

        state.v.mul_(beta2).addcmul_(grad, grad, value=1.0-beta2)

        bias_correction1 = 1.0 - (beta1 ** t)
        bias_correction2 = 1.0 - (beta2 ** t)

        m_hat = state.m / bias_correction1
        v_hat = state.v / bias_correction2

        if weight_decay != 0:
            p.mul_(1.0 - lr * weight_decay)

        step_term = m_hat / (v_hat.sqrt() + eps)
        p.sub_(step_term, alpha=lr)

    return AdamWState(m=state.m, v=state.v, t=t)

In [22]:
def adamw_step_many_(
    params: list[torch.Tensor],
    grads: list[torch.Tensor],
    states: list[AdamWState],
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> list[AdamWState]:
    """
    Apply AdamW to many parameters.

    Requirements:
    - len(params) == len(grads) == len(states)
    - Update each param in-place.
    - Return the list of updated states.
    """
    assert len(params) == len(grads) == len(states), "Mismatched argument lengths."

    updated_states = []
    for p, g, state in zip(params, grads, states, strict=True):
        new_state = adamw_step_(
            p=p,
            grad=g,
            state=state,
            lr=lr,
            betas=betas,
            eps=eps,
            weight_decay=weight_decay,
        )
        updated_states.append(new_state)

    return updated_states

## Training basics

A minimal training step follows the same pattern almost everywhere:

1) set model to train mode
2) reset gradients
3) forward pass
4) compute loss
5) backward pass
6) step optimizer

In this exercise you’ll implement a single MSE training step using a standard PyTorch optimizer.
Return a Python float loss value.

In [23]:
def train_step_mse(
    model: nn.Module,
    batch: tuple[torch.Tensor, torch.Tensor],
    optimizer: torch.optim.Optimizer,
) -> float:
    """
    One MSE train step using standard torch optimizer.
    """
    model.train()
    optimizer.zero_grad()
    inputs, targets = batch

    predictions = model(inputs)
    loss = nn.functional.mse_loss(predictions, targets)

    loss.backward()
    optimizer.step()

    return loss.item()


## Parameter initialization

Initialization matters because it controls signal and gradient scales at the start of training.

### Fan-in / fan-out
- `fan_in`: number of input connections to a unit
- `fan_out`: number of output connections from a unit

For a Linear layer weight of shape `(out_features, in_features)`:
- `fan_in = in_features`
- `fan_out = out_features`

### Common schemes
- **Xavier / Glorot** (often good for tanh / linear-ish nets):
  keeps variance stable across layers when activations are roughly symmetric.
- **Kaiming / He** (often good for ReLU-like nets):
  accounts for the fact that ReLU zeroes out about half the inputs.

In this section you’ll implement Xavier uniform and Kaiming uniform and use them to initialize `nn.Linear`.
We also always zero the bias unless explicitly told otherwise.

In [24]:
def fan_in_fan_out(weight: torch.Tensor) -> tuple[int, int]:
    """Compute (fan_in, fan_out) for a weight tensor."""
    dimensions = weight.ndim
    if dimensions < 2:
        num_input_fmaps = weight.numel()
        num_output_fmaps = weight.numel()
        return num_input_fmaps, num_output_fmaps

    num_input_fmaps = weight.size(1)
    num_output_fmaps = weight.size(0)

    receptive_field_size = 1
    if dimensions > 2:
        for s in weight.shape[2:]:
            receptive_field_size *= s

    fan_in = num_input_fmaps * receptive_field_size
    fan_out = num_output_fmaps * receptive_field_size

    return fan_in, fan_out

In [25]:
import math

def xavier_uniform_(weight: torch.Tensor, gain: float = 1.0) -> torch.Tensor:
    """
    In-place Xavier/Glorot uniform init:
      bound = gain * sqrt(6 / (fan_in + fan_out))
      U(-bound, bound)
    """
    fan_in, fan_out = fan_in_fan_out(weight)
    
    bound = gain * math.sqrt(6.0 / (fan_in + fan_out))
    
    with torch.no_grad():
        weight.uniform_(-bound, bound)
        
    return weight

In [26]:
def kaiming_uniform_(weight: torch.Tensor, nonlinearity: str = "relu") -> torch.Tensor:
    """
    In-place Kaiming/He uniform init.

    Follow this common choice:
      gain = sqrt(2) for ReLU
      std = gain / sqrt(fan_in)
      bound = sqrt(3) * std
      U(-bound, bound)
    """
    fan_in, fan_out = fan_in_fan_out(weight)
        
    gain = 2**(1/2)
    std = gain / (fan_in)**(1/2)
    bound = 3**(1/2) * std
    
    with torch.no_grad():
        weight.uniform_(-bound, bound)
        
    return weight

In [27]:
def init_linear_(layer: nn.Linear, scheme: str = "xavier") -> nn.Linear:
    """
    Initialize an nn.Linear in-place.

    scheme:
      - "xavier"
      - "kaiming_relu"
      - "zero" (weights and bias = 0)
    """
    with torch.no_grad():
        if scheme == "zero":
            layer.weight.zero_()
            if layer.bias is not None:
                layer.bias.zero_()

        elif scheme == "xavier":
            xavier_uniform_(layer.weight, gain=1.0)
            if layer.bias is not None:
                layer.bias.zero_()

        elif scheme == "kaiming_relu":
            kaiming_uniform_(layer.weight, gain=1.0)
            if layer.bias is not None:
                layer.bias.zero_()

        else:
            raise ValueError(f"Unknown initialization scheme: '{scheme}'")

    return layer